# J2-01 · Persistance : checkpoints, threads et voyage dans le temps

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 2 · créneau 08h30–09h45 · ⏱️ ~1h15 (dont 45 min de pratique)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Donner une mémoire à l'agent avec un **checkpointer** · 2. Isoler les conversations par **`thread_id`** · 3. Lire l'état et son historique (`get_state`, `get_state_history`) · 4. **Rejouer** et **bifurquer** depuis un checkpoint · 5. Mesurer ce qu'un checkpoint contient au regard du **RGPD**, et l'effacer |
| 🧱 **Fil rouge** | L'agent support du jalon J1 (`helpdesk_ops.jalons.j1`) devient **stateful** |
| ✅ **Prérequis** | Jour 1 terminé ; stack démarrée |
| 📦 **Versions** | `langgraph 1.2` · `langgraph-checkpoint-sqlite 3.1` · `langfuse 4.15`, figées par `uv.lock` |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
from helpdesk_ops.jalons.j1 import construire_support, repondre   # le jalon de fin de J1 (corrigés Ex1.1 + Ex1.2)

def demander(graphe, question: str, thread_id: str | None = None) -> str:
    """Un tour de conversation, tracé ; thread_id=None : sans mémoire."""
    config = {"configurable": {"thread_id": thread_id}} if thread_id else {}
    with propagate_attributes(session_id=thread_id or "j2-01-sans-memoire", tags=["jour-2", "persistance"]):
        return repondre(graphe, question, callbacks=[langfuse_handler], config=config)

## 2. Un agent sans mémoire

### [Théorie] Chaque `invoke` repart de zéro

Sans checkpointer, l'état du graphe vit le temps d'un `invoke`. Le deuxième message ne sait rien du premier : pour l'utilisateur, l'agent est **amnésique**.

In [ ]:
amnesique = construire_support(llm)
print("1 →", demander(amnesique, "Bonjour, le serveur srv-erp-02 est-il en ligne ?"))
print("2 →", demander(amnesique, "Et de quel serveur venons-nous de parler ?"))

**👀 Ce que vous devez observer :** à la deuxième question, l'agent ne sait pas (ou pire, invente). Sans mémoire, impossible de dire « et pour l'autre serveur ? ».

> ⚠️ **Effet de bord à retenir :** le garde-fou par règles de J1 juge chaque message **isolément**. Une relance comme « et l'autre ? » ne contient aucun mot-clé support : elle serait refusée, même avec la mémoire. Un garde-fou de conversation doit regarder le contexte (piste d'amélioration, hors programme de cette session).

## 3. Le checkpointer : une sauvegarde à chaque étape

### [Théorie] Les sauvegardes d'un jeu vidéo

*Analogie :* le jeu sauvegarde automatiquement à chaque niveau franchi. On peut reprendre la partie, revenir à une sauvegarde précédente, ou repartir d'une sauvegarde pour essayer autre chose.

LangGraph fait pareil : à la fin de **chaque super-step** (chaque passage de nœud), il écrit un **checkpoint** (l'état complet + la suite prévue). Les checkpoints d'une même conversation partagent un **`thread_id`**.

| Checkpointer | Stockage | Usage |
|---|---|---|
| `InMemorySaver` | mémoire du processus | tests, démos (perdu au redémarrage) |
| `SqliteSaver` / `AsyncSqliteSaver` | un fichier SQLite | poste unique, formation |
| `PostgresSaver` | PostgreSQL | production (plusieurs instances) |

> 💡 **Le « Tips » du Formateur :** `thread_id` = **une conversation**, pas un utilisateur. Un même utilisateur peut avoir dix conversations ; les mélanger dans un seul thread, c'est mélanger leurs contextes (et leurs données personnelles).

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

avec_memoire = construire_support(llm, checkpointer=InMemorySaver())

print("A1 →", demander(avec_memoire, "Bonjour, le serveur srv-erp-02 est-il en ligne ?", thread_id="j2-01-conv-A"))
print("A2 →", demander(avec_memoire, "Et ce serveur, quel est son nom exact déjà ? Réponds juste par le nom du serveur.", thread_id="j2-01-conv-A"))
print("B1 →", demander(avec_memoire, "Quel serveur avons-nous évoqué ? Réponds juste par le nom du serveur.", thread_id="j2-01-conv-B"))

**👀 Ce que vous devez observer :** la conversation **A** se souvient de `srv-erp-02` ; la conversation **B**, nouvelle, n'en sait rien. Même graphe, même checkpointer : c'est le `thread_id` qui isole.

Dans Langfuse, **Sessions** : `j2-01-conv-A` regroupe 2 traces. On a aligné `session_id` sur `thread_id` : **une conversation = un thread = une session**.

## 4. Lire l'état : `get_state` et `get_state_history`

### [Théorie] Le `StateSnapshot`

| Champ | Contenu |
|---|---|
| `values` | l'état (ici : `messages`, `decision`) |
| `next` | le(s) nœud(s) à exécuter ensuite ; `()` = terminé |
| `config` | contient le `checkpoint_id` : l'adresse de cette sauvegarde |
| `metadata` | `step`, `source` (`input`, `loop`, `update`…), écritures du step |
| `tasks` | les tâches en attente (et les interruptions, J2-02) |

In [ ]:
config_A = {"configurable": {"thread_id": "j2-01-conv-A"}}
etat = avec_memoire.get_state(config_A)
print("next       :", etat.next)
print("messages   :", len(etat.values["messages"]))
print("checkpoint :", etat.config["configurable"]["checkpoint_id"])

print("\nHistorique (du plus récent au plus ancien) :")
for snap in avec_memoire.get_state_history(config_A):
    dernier = snap.values.get("messages", [])[-1:] or [None]
    print(f"  step {snap.metadata['step']:>2} | next={str(snap.next):<16} | dernier message : "
          f"{type(dernier[0]).__name__ if dernier[0] else '-':<12} {str(getattr(dernier[0], 'content', ''))[:45]!r}")

**👀 Ce que vous devez observer :** un checkpoint par étape (`garde_fou`, `agent`, `tools`, `agent`…), pour **les deux tours**. Le step `-1` est l'entrée brute. C'est cet historique qui rend possibles la reprise après panne, la validation humaine (J2-02) et le voyage dans le temps.

## 5. `SqliteSaver` : survivre à un redémarrage

### [Théorie] Un fichier, deux tables

`SqliteSaver` écrit dans un fichier : `checkpoints` (les états) et `writes` (les écritures intermédiaires). Le fichier est rangé dans `Orchestration/data/checkpoints/` (créé au besoin) et **ignoré par Git** : il contient des conversations.

> 💡 **Le « Tips » du Formateur :** `check_same_thread=False` est nécessaire car LangGraph peut écrire depuis un autre thread Python que celui qui a ouvert la connexion. En asynchrone (J2-04), on utilise `AsyncSqliteSaver`.

In [ ]:
import sqlite3

from langgraph.checkpoint.sqlite import SqliteSaver

from helpdesk_ops.persistance import chemin_checkpoints

CHEMIN = chemin_checkpoints("j2_01_checkpoints.sqlite")
connexion = sqlite3.connect(CHEMIN, check_same_thread=False)
saver = SqliteSaver(connexion)

support = construire_support(llm, checkpointer=saver)
print("1 →", demander(support, "Je suis Julie Martin (julie.martin@nordtech.example). srv-erp-02 ne répond plus depuis ce matin.",
                       thread_id="j2-01-julie"))
print(f"\nFichier : {CHEMIN.name} ({CHEMIN.stat().st_size / 1024:.0f} Ko)")

In [ ]:
# « Redémarrage » : on jette le graphe et la connexion, on repart du fichier
del support, saver
connexion.close()

connexion = sqlite3.connect(CHEMIN, check_same_thread=False)
saver = SqliteSaver(connexion)
support = construire_support(llm, checkpointer=saver)
print("2 →", demander(support, "Rappelle-moi le serveur concerné par mon problème. Réponds juste par son nom.", thread_id="j2-01-julie"))

**👀 Ce que vous devez observer :** après le « redémarrage », la conversation reprend : le serveur `srv-erp-02` est retrouvé depuis le fichier.

> ⚠️ **Vu en préparant ce notebook :** une réponse se terminait par `[TOOL_CALLS]chercher_kb{"question": …}` **en texte**. Le modèle a voulu appeler un outil, mais dans un format que le parseur n'a pas reconnu : la boucle s'arrête et l'utilisateur voit du charabia. C'est un défaut typique des petits modèles locaux ; on le **mesurera** en J4 (taux d'appels d'outils valides) et on verra au red-team du J4 comment un **contrôle de sortie** le filtre.

## 6. Voyage dans le temps : rejouer et bifurquer

### [Théorie] Repartir d'une sauvegarde

- **Rejouer** : `graphe.invoke(None, config_d_un_checkpoint)` relance l'exécution **à partir** de ce checkpoint.
- **Bifurquer** (*fork*) : `graphe.update_state(config, valeurs)` crée un **nouveau** checkpoint à partir d'un ancien, avec un état modifié ; on relance depuis ce fork. L'historique d'origine reste intact.

*Cas d'usage :* rejouer une conversation qui a mal tourné en corrigeant un message ; déboguer un agent ; expliquer une décision à un auditeur.

In [ ]:
from langchain_core.messages import HumanMessage

config_julie = {"configurable": {"thread_id": "j2-01-julie"}}
historique = list(support.get_state_history(config_julie))

# Le checkpoint juste AVANT le garde-fou du 1er tour : le message de Julie est dans l'état, rien n'a encore tourné
avant_tour_1 = [s for s in historique if s.next == ("garde_fou",)][-1]
message_initial = avant_tour_1.values["messages"][-1]
print("Message d'origine :", message_initial.content[:80], "…")

# Bifurcation : même id de message → add_messages le REMPLACE (vu en J1-03)
corrige = HumanMessage("srv-mail-01 ne répond plus depuis ce matin.", id=message_initial.id)
fork = support.update_state(avant_tour_1.config, {"messages": [corrige]})

with propagate_attributes(session_id="j2-01-julie", tags=["jour-2", "time-travel"]):
    etat_fork = support.invoke(None, {**fork, "callbacks": [langfuse_handler], "run_name": "fork", "recursion_limit": 10})
print("Réponse sur la branche corrigée :", etat_fork["messages"][-1].content)

**👀 Ce que vous devez observer :** la nouvelle branche répond à propos de `srv-mail-01`. `get_state(config_julie)` renvoie désormais la **dernière** branche ; les checkpoints d'origine existent toujours dans l'historique.

> ⚠️ **Rejouer n'est pas gratuit :** les nœuds relancés rappellent le LLM… et les **outils**. Un outil qui écrit (créer un ticket, envoyer un mail) rejoué deux fois, c'est un doublon. D'où l'**idempotence**, au cœur de J2-02.

## 7. 🔒 RGPD : ce que contient un checkpoint

### [Théorie] Le masquage Langfuse ne protège pas les checkpoints

En J1, on a masqué e-mails et IBAN **avant l'envoi vers Langfuse**. Mais le checkpoint, lui, stocke **l'état brut** : tous les messages, en clair. Vérifions dans le fichier.

In [ ]:
lignes = connexion.execute("SELECT COUNT(*), COUNT(DISTINCT thread_id) FROM checkpoints").fetchone()
print(f"{lignes[0]} checkpoints, {lignes[1]} thread(s)")

fuites = connexion.execute(
    "SELECT COUNT(*) FROM checkpoints WHERE thread_id = ? AND instr(checkpoint, ?) > 0",
    ("j2-01-julie", "julie.martin@nordtech.example".encode()),
).fetchone()[0]
print(f"Checkpoints du thread 'j2-01-julie' contenant l'e-mail EN CLAIR : {fuites}")

| Obligation RGPD | Pour les checkpoints |
|---|---|
| Minimisation (art. 5.1.c) | ne pas stocker plus que nécessaire. ⚠️ `saver.prune(threads, strategy="keep_latest")` est documenté mais **non implémenté** par `SqliteSaver` 3.1 (`NotImplementedError`) : on supprime les conversations terminées plutôt que de les élaguer |
| Limitation de conservation (art. 5.1.e) | purge **planifiée** des conversations inactives (`helpdesk_ops.persistance.purger_threads`) |
| Droit à l'effacement (art. 17) | `saver.delete_thread(thread_id)` |
| Sécurité (art. 32) | fichier hors Git, droits restreints, disque chiffré ; en production, base chiffrée et accès journalisés |

> 💡 **Le « Tips » du Formateur :** la question à poser à l'équipe n'est pas « peut-on supprimer ? » mais « **combien de temps** garde-t-on, **qui** déclenche la purge, et **où est-ce écrit** ? ». Une durée sans automate ne sera jamais appliquée.

In [ ]:
from datetime import timedelta

from helpdesk_ops.persistance import purger_threads

# 1. Simulation : quels threads seraient purgés avec une conservation de 30 jours ? (ici : aucun, tout est récent)
print("Purge à 30 jours (simulation) :", purger_threads(saver, timedelta(days=30), simulation=True))

# 2. Minimisation : prune() existe dans l'API… mais pas pour SQLite. On vérifie plutôt que de supposer.
try:
    saver.prune(["j2-01-julie"], strategy="keep_latest")
except NotImplementedError:
    print("prune(keep_latest) : non implémenté par SqliteSaver 3.1 → on supprimera la conversation entière")

# 3. Droit à l'effacement : Julie demande la suppression de sa conversation
saver.delete_thread("j2-01-julie")
print("Après delete_thread :", connexion.execute(
    "SELECT COUNT(*) FROM checkpoints WHERE thread_id = 'j2-01-julie'").fetchone()[0], "checkpoint")
print("État relu :", support.get_state(config_julie).values)

**👀 Ce que vous devez observer :** l'e-mail de Julie est présent **en clair** dans plusieurs checkpoints ; `prune` n'est pas disponible avec SQLite ; `delete_thread` supprime la conversation (checkpoints **et** écritures) : l'état relu est vide. Leçon d'architecte : une API documentée n'est pas une API implémentée pour **votre** backend. Testez-la.

**Question pour la salle :** Julie a exercé son droit à l'effacement. Sa conversation est-elle **vraiment** effacée partout ? *(Indice : les traces Langfuse, masquées mais pas vides, et dont le nom « Julie Martin » n'était pas masqué ; les sauvegardes du fichier SQLite ; les journaux applicatifs…)*

## 8. 🧪 Mini-défi (10 min) : le droit d'accès

L'article 15 du RGPD donne à chacun le droit d'obtenir **une copie** de ses données.

1. Créez une conversation `j2-01-defi-<initiales>` de deux tours.
2. Écrivez `exporter_conversation(graphe, thread_id) -> list[dict]` qui renvoie `[{"role": …, "contenu": …}]` pour les messages humains et les réponses de l'agent (sans les appels d'outils).
3. Discussion : faut-il inclure les **résultats d'outils** ? les **décisions du garde-fou** ?

In [ ]:
# À vous de jouer 👇

In [ ]:
connexion.close()
langfuse.flush()
print("✅ Connexion SQLite fermée, observations envoyées")

## 🧭 Récap

| Vous savez maintenant… | API |
|---|---|
| Donner une mémoire à un graphe | `compile(checkpointer=InMemorySaver() / SqliteSaver(conn))` |
| Isoler les conversations | `config={"configurable": {"thread_id": …}}` |
| Lire l'état et son historique | `get_state`, `get_state_history` |
| Rejouer / bifurquer | `invoke(None, config_checkpoint)`, `update_state(config, valeurs)` |
| Purger, effacer | `purger_threads` (planifiée), `delete_thread` (art. 17) |

**Prochain notebook, J2-02 :** l'agent s'arrête et attend une **validation humaine** avant de créer un ticket P1, grâce aux checkpoints.

### ✅ Quiz express

1. Que représente un `thread_id` ?
2. Que se passe-t-il pour un `InMemorySaver` au redémarrage du kernel ?
3. Quelle différence entre rejouer et bifurquer ?
4. Le masquage PII de Langfuse protège-t-il les checkpoints ?

<details>
<summary>Réponses</summary>

1. Une conversation (pas un utilisateur) : tous les checkpoints d'une même conversation.
2. Tout est perdu : il vit dans la mémoire du processus. `SqliteSaver` (ou Postgres) persiste.
3. Rejouer relance depuis un checkpoint existant ; bifurquer crée d'abord un nouveau checkpoint à l'état modifié (`update_state`), l'historique d'origine restant intact.
4. Non : il ne s'applique qu'à l'envoi vers Langfuse. Les checkpoints stockent l'état en clair : conservation limitée, purge, effacement.
</details>